# Transformer - Fine-Tuning

## Importing the required libraries and functions. Defining global variables.

In [ ]:
import random
import sys

sys.path.append("..")

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
)

LABELS = {1: "World", 2: "Sports", 3: "Business", 4: "Sci/Tech"}

TRANSFORMER_CONFIG = {
    "model_name": "distilbert-base-uncased",
    "max_len": 64,
    "batch_size": 32,
    "lr": 2e-5,
    "epochs": 4,
    "patience": 2,
    "warmup_ratio": 0.1,
    "num_classes": 4,
    "weight_decay": 0.01,
}

In [ ]:
def set_seed(seed: int = 42) -> None:
    """
    Set random seeds for reproducibility. Defaults to 42.

    Args:
        seed (int): Seed value.

    Returns:
        None: Sets the seed to a given value or 42.
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed(42)

In [ ]:
if torch.cuda.is_available():
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")
    device = torch.device("cuda")
else:
    print("Using CPU")
    device = torch.device("cpu")

## Defining a custom dataset class

In [ ]:
class AGNewsDataset(Dataset):
    """
    A custom dataset class for the AG News dataset.
    """
    
    def __init__(self, encodings: dict, labels: list) -> None:
        """
        Constructor for the AGNewsDataset class.

        Args:
            encodings (dict): A dictionary containing the tokenized input data.
            labels (list): A list of labels corresponding to the input data.

        Returns:
            None: Initializes the dataset with the provided encodings and labels.
        """
        self.encodings = encodings
        self.labels = list(labels)

    def __len__(self) -> int:
        """
        Returns the length of the dataset.

        Args:
            None

        Returns:
            int: The number of samples in the dataset.
        """
        return len(self.labels)

    def __getitem__(self, idx: int) -> dict:
        """
        Returns a single sample from the dataset at the specified index.

        Args:
            idx (int): The index of the sample to retrieve.

        Returns:
            dict: A dictionary containing the input IDs, attention mask, and label for the specified sample.
        """
        return {
            "input_ids": torch.tensor(self.encodings["input_ids"][idx]),
            "attention_mask": torch.tensor(self.encodings["attention_mask"][idx]),
            "label": torch.tensor(self.labels[idx], dtype=torch.long),
        }

## Loading and preprocessing the data

In [ ]:
splits = {'train': 'train.jsonl', 'test': 'test.jsonl'}
df_train = pd.read_json("hf://datasets/sh0416/ag_news/" + splits["train"], lines=True)
test_dataset = pd.read_json("hf://datasets/sh0416/ag_news/" + splits["test"], lines=True)

train_dataset, val_dataset = train_test_split(
    df_train,
    test_size=0.1,
    random_state=42,
    stratify=df_train["label"]
)

X_train = train_dataset["title"] + " " + train_dataset["description"]
y_train = train_dataset["label"]

X_val = val_dataset["title"] + " " + val_dataset["description"]
y_val = val_dataset["label"]

X_test = test_dataset["title"] + " " + test_dataset["description"]
y_test = test_dataset["label"]

## Tokenize the text data

In [ ]:
def to_zero_indexed(series: pd.Series) -> list:
    """
    Convert a pandas Series of labels to zero-indexed format.

    Args:
        series (pd.Series): A pandas Series containing the labels.

    Returns:
        list: A list of zero-indexed labels.
    """
    return (series - 1).tolist()

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(TRANSFORMER_CONFIG["model_name"])

train_encodings = tokenizer(
    X_train.tolist(),
    truncation=True,
    padding=True,
    max_length=TRANSFORMER_CONFIG["max_len"],
)

val_encodings = tokenizer(
    X_val.tolist(),
    truncation=True,
    padding=True,
    max_length=TRANSFORMER_CONFIG["max_len"],
)

test_encodings = tokenizer(
    X_test.tolist(),
    truncation=True,
    padding=True,
    max_length=TRANSFORMER_CONFIG["max_len"],
)

train_labels_0 = to_zero_indexed(y_train)
val_labels_0   = to_zero_indexed(y_val)
test_labels_0  = to_zero_indexed(y_test)

tokenized_train = AGNewsDataset(train_encodings, train_labels_0)
tokenized_val   = AGNewsDataset(val_encodings, val_labels_0)
tokenized_test  = AGNewsDataset(test_encodings, test_labels_0)

## Training and saving the model

In [ ]:
def compute_metrics(eval_pred: tuple) -> dict:
    """
    Compute accuracy and macro F1 score for model evaluation.
    
    Args:
        eval_pred (tuple): A tuple containing the model's predictions and true labels.

    Returns:
        dict: A dictionary containing accuracy and macro F1 score.
    """
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average="macro")
    
    return {"accuracy": acc, "macro_f1": f1}

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    TRANSFORMER_CONFIG["model_name"],
    num_labels=TRANSFORMER_CONFIG["num_classes"],
)
model = model.to(device)

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    logging_strategy="epoch",
    save_strategy="epoch",
    learning_rate=TRANSFORMER_CONFIG["lr"],
    per_device_train_batch_size=TRANSFORMER_CONFIG["batch_size"],
    per_device_eval_batch_size=TRANSFORMER_CONFIG["batch_size"],
    num_train_epochs=TRANSFORMER_CONFIG["epochs"],
    weight_decay=TRANSFORMER_CONFIG["weight_decay"],
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

trainer.train()
trainer.save_model("./distilbert_agnews")
tokenizer.save_pretrained("./distilbert_agnews")

## Evaluating the model

In [ ]:
def learning_curves(log_history: list) -> None:
    """
    Plot learning curves for training and evaluation metrics.

    Args:
        log_history (list): A list of dictionaries containing training and evaluation metrics.

    Returns:
        None: Displays the learning curves for loss, accuracy, and macro F1 score.
    """
    train_loss = [
        x["loss"] for x in log_history if "loss" in x and "eval_loss" not in x
    ]
    eval_loss = [x["eval_loss"] for x in log_history if "eval_loss" in x]
    eval_acc = [x["eval_accuracy"] for x in log_history if "eval_accuracy" in x]
    eval_f1 = [x["eval_macro_f1"] for x in log_history if "eval_macro_f1" in x]
    epochs = range(1, len(eval_loss) + 1)

    train_epochs = range(1, len(train_loss) + 1)
    eval_epochs = range(1, len(eval_loss) + 1)

    _, axes = plt.subplots(1, 3, figsize=(15, 4))

    axes[0].plot(train_epochs, train_loss, label="Train Loss")
    axes[0].plot(eval_epochs, eval_loss, label="Val Loss")
    axes[0].set_title("Loss")
    axes[0].legend()

    axes[1].plot(epochs, eval_acc)
    axes[1].set_title("Val Accuracy")

    axes[2].plot(epochs, eval_f1)
    axes[2].set_title("Val Macro-F1")

    plt.tight_layout()
    plt.show()

In [ ]:
eval_results = trainer.evaluate()
print(f"Evaluation Results: {eval_results}")

test_results = trainer.predict(tokenized_test)
test_preds = np.argmax(test_results.predictions, axis=-1)
test_labels_list = test_results.label_ids

test_acc = accuracy_score(test_labels_list, test_preds)
test_f1 = f1_score(test_labels_list, test_preds, average="macro")
print(f"Test Accuracy: {test_acc:.4f}")
print(f"Test Macro-F1: {test_f1:.4f}")

In [ ]:
cm = confusion_matrix(test_labels_list, test_preds)
print("Confusion Matrix:")
print(cm)

In [ ]:
log_history = trainer.state.log_history
learning_curves(log_history)

# Error analysis

In [ ]:
def collect_errors(texts: pd.Series, true_labels: np.ndarray, pred_labels: np.ndarray, label_map: dict, n: int = 20) -> pd.DataFrame:
    """
    Returns a DataFrame of misclassified examples.

    Args:
        texts (pd.Series): A Series of text examples.
        true_labels (np.ndarray): An array of true labels.
        pred_labels (np.ndarray): An array of predicted labels.
        label_map (dict): A dictionary mapping label indices to their string representations.
        n (int): The number of misclassified examples to return.

    Returns:
        pd.DataFrame: A DataFrame containing the misclassified examples.
    """
    texts_list = texts.tolist() if hasattr(texts, "tolist") else list(texts)
    rows = []
    for i, (true, pred) in enumerate(zip(true_labels, pred_labels)):
        if true != pred:
            rows.append({
                "text":       texts_list[i],
                "true_label": label_map[true + 1], 
                "pred_label": label_map[pred + 1],
            })
    return pd.DataFrame(rows).head(n)

In [ ]:
error_df = collect_errors(X_test, test_labels_list, test_preds, LABELS, n=20)

print("\nFirst 20 misclassified examples (DistilBERT):\n")
pd.set_option("display.max_colwidth", 120)
print(error_df.to_string(index=False))